In [ ]:
!pip install nltk
!pip install tqdm
!pip install rank-bm25 rouge-score nltk
!pip install --upgrade datasets

from nltk.tokenize import word_tokenize, sent_tokenize
from collections import defaultdict, Counter
from sklearn.metrics.pairwise import cosine_similarity
from datasets import Dataset, DatasetDict, load_dataset
from huggingface_hub import login
from tqdm import tqdm
from sklearn.metrics import cohen_kappa_score
from rank_bm25 import BM25Okapi
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.tokenize import word_tokenize
from scipy.stats import spearmanr, kendalltau
from itertools import combinations
from sklearn.metrics.pairwise import cosine_similarity

import json
import pandas as pd
import nltk
import matplotlib.pyplot as plt
import statistics
import numpy as np
import random
import re
import seaborn as sns
import re
import nltk

In [ ]:
HF_L = "XXX"
login(token=HF_L)

dataset = load_dataset("Ramitha/squad-600-length-results")
df = pd.DataFrame(dataset['rawcases'])

In [3]:
def get_cosine_similarity(emb1, emb2):
    emb1 = emb1.reshape(1, -1) if emb1.ndim == 1 else emb1
    emb2 = emb2.reshape(1, -1) if emb2.ndim == 1 else emb2
    similarity = cosine_similarity(emb1, emb2)
    return similarity[0, 0]

def get_case_alignment(case_embs, case_base):
    emb1, emb2 = case_embs
    emb1 = emb1.reshape(1, -1) if emb1.ndim == 1 else emb1
    emb2 = emb2.reshape(1, -1) if emb2.ndim == 1 else emb2
    alignment_scores = []
    for past_case in case_base:
        past_prob_emb, past_solution_emb = past_case
        past_prob_emb = past_prob_emb.reshape(1, -1) if past_prob_emb.ndim == 1 else past_prob_emb
        past_solution_emb = past_solution_emb.reshape(1, -1) if past_solution_emb.ndim == 1 else past_solution_emb
        prob_similarity = cosine_similarity(emb1, past_prob_emb)
        solution_similarity = cosine_similarity(emb2, past_solution_emb)
        alignment_score = (prob_similarity + solution_similarity) / 2.0
        alignment_scores.append(alignment_score)
    return (sum(alignment_scores) / len(alignment_scores))[0][0]

def get_weighted_case_alignment(case_embs, case_base):
    emb1, emb2 = case_embs
    emb1 = emb1.reshape(1, -1) if emb1.ndim == 1 else emb1
    emb2 = emb2.reshape(1, -1) if emb2.ndim == 1 else emb2

    alignment_scores = []
    weights = []
    for past_case in case_base:
        past_prob_emb, past_solution_emb = past_case
        past_prob_emb = past_prob_emb.reshape(1, -1) if past_prob_emb.ndim == 1 else past_prob_emb
        past_solution_emb = past_solution_emb.reshape(1, -1) if past_solution_emb.ndim == 1 else past_solution_emb
        prob_similarity = cosine_similarity(emb1, past_prob_emb)[0][0]
        solution_similarity = cosine_similarity(emb2, past_solution_emb)[0][0]
        alignment_score = (prob_similarity + solution_similarity) / 2.0
        alignment_scores.append(alignment_score)
        weights.append(prob_similarity)
    total_weight = sum(weights)
    if total_weight == 0:
        return 0
    normalized_weights = [w / total_weight for w in weights]
    weighted_alignment_score = sum(a * w for a, w in zip(alignment_scores, normalized_weights))
    return weighted_alignment_score


def get_tuned_weighted_case_alignment(case_embs, case_base):
    emb1, emb2 = case_embs
    emb1 = emb1.reshape(1, -1) if emb1.ndim == 1 else emb1
    emb2 = emb2.reshape(1, -1) if emb2.ndim == 1 else emb2

    alignment_scores = []
    weights = []

    for past_case in case_base:
        past_prob_emb, past_solution_emb = past_case
        past_prob_emb = past_prob_emb.reshape(1, -1) if past_prob_emb.ndim == 1 else past_prob_emb
        past_solution_emb = past_solution_emb.reshape(1, -1) if past_solution_emb.ndim == 1 else past_solution_emb

        prob_similarity = cosine_similarity(emb1, past_prob_emb)[0][0]  # ∈ [0,1]
        solution_similarity = cosine_similarity(emb2, past_solution_emb)[0][0]  # ∈ [0,1]

        alignment_score = (
            1 * prob_similarity + 9 * solution_similarity
        ) / (1 + 9)

        alignment_scores.append(alignment_score)
        weights.append(prob_similarity)
    alignment_scores = np.array(alignment_scores)
    weights = np.array(weights)
    if len(weights) == 0 or np.sum(weights) == 0:
        return 0.0

    epsilon = 1e-6
    inv_weights = 1 / (1 - weights + epsilon)
    normalized_inv_weights = inv_weights / np.sum(inv_weights)
    weighted_alignment_score = np.sum(alignment_scores * normalized_inv_weights)
    return float(weighted_alignment_score)

def get_question_alignment(case_embs, case_base):
    emb1, _ = case_embs
    emb1 = emb1.reshape(1, -1) if emb1.ndim == 1 else emb1
    alignment_scores = []
    for past_case in case_base:
        past_prob_emb, _ = past_case
        past_prob_emb = past_prob_emb.reshape(1, -1) if past_prob_emb.ndim == 1 else past_prob_emb
        prob_similarity = cosine_similarity(emb1, past_prob_emb)
        alignment_scores.append(prob_similarity)
    return (sum(alignment_scores) / len(alignment_scores))[0][0]

GENERATE THE ALIGNMENT SCORES FOR ILR, WILR and TUNED-WILR

In [ ]:
models = ["llama", "falcon", "gemma", "mistral"]

def clean_embedding(x):
    x = np.array(x, dtype=np.float64)
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
    return x
    
for index, row in tqdm(df.iterrows(), total=len(df)):
    for model in models:

        q_emb = clean_embedding(json.loads(row[f"question_{model}_emb"]))
        a_emb = clean_embedding(json.loads(row[f"answerGenerated_{model}_emb"]))

        case_ = [q_emb, a_emb]

        case_base = []
        for other_model in models:
            if other_model == model:
                continue

            q_ = clean_embedding(json.loads(row[f"question_answerGenerated_{other_model}_{model}_emb"]))
            a_ = clean_embedding(json.loads(row[f"reverse_answer_answerGenerated_{other_model}_{model}_emb"]))

            case_base.append([q_, a_])

        df.at[index, f"ILRAlign_{model}"] = get_case_alignment(case_, case_base)
        df.at[index, f"WILRAlign_{model}"] = get_weighted_case_alignment(case_, case_base)
        df.at[index, f"WILRAlign_tuned_{model}"] = get_tuned_weighted_case_alignment(case_, case_base)

hf_dataset = DatasetDict({
    'rawcases': Dataset.from_pandas(df)
})
hf_dataset.push_to_hub("Ramitha/squad-600-length-results")
